In [35]:
import requests
import json
import os

In [36]:
folder = "test_endpoints"
os.makedirs(folder, exist_ok=True)

In [37]:
def _infer_schema(value): 
    if value is None: 
        return {"type": "null"} 
    if isinstance(value, bool): 
        return {"type": "boolean"} 
    if isinstance(value, int): 
        return {"type": "integer"} 
    if isinstance(value, float): 
        return {"type": "number"} 
    if isinstance(value, str): 
        return {"type": "string"} 
    if isinstance(value, list): 
        schema = {
            "type": "array"
        } 
        if value: 
            item_schemas = [_infer_schema(item) for item in value] 
            if all(item_schema == item_schemas[0] for item_schema in item_schemas):
                schema["items"] = item_schemas[0]
            else:
                schema["items"] = item_schemas
        return schema 
    if isinstance(value, dict): 
        properties = {} 
        for key, child_value in value.items(): 
            properties[key] = _infer_schema(child_value) 
        return { "type": "object", "properties": properties } 
    return { "type": "unknown" }

def explore_endpoint(
    url_v,
    endpoint,
    params=None,
    timeout=30
):
    BASE_URL = {
        "1": "https://statsapi.mlb.com/api/v1/",
        "1.1": "https://statsapi.mlb.com/api/v1.1/"
    }
    if endpoint.startswith("http://") or endpoint.startswith("https://"):
        url = endpoint
    else:
        endpoint = endpoint.lstrip("/")
        url = f"{BASE_URL[url_v]}/{endpoint}"

    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )

    response.raise_for_status()

    data = response.json()

    return {
        "schema": _infer_schema(data),
        "response": data
    }

In [38]:
def compare_responses(response1, response2):
    if isinstance(response1, dict) and isinstance(response2, dict):
        keys = {}

        all_keys = response1.keys() | response2.keys()

        for key in all_keys:
            in_1 = key in response1
            in_2 = key in response2

            if in_1 and not in_2:
                keys[key] = False
            elif not in_1 and in_2:
                keys[key] = "added"
            else:
                keys[key] = compare_responses(
                    response1[key],
                    response2[key]
                )

        return keys

    if isinstance(response1, list) and isinstance(response2, list):
        results = []

        for i, (item1, item2) in enumerate(zip(response1, response2)):
            results.append(
                compare_responses(item1, item2)
            )

        if len(response1) > len(response2):
            results.extend(
                [False] * (len(response1) - len(response2))
            )

        elif len(response2) > len(response1):
            results.extend(
                ["added"] * (len(response2) - len(response1))
            )

        return results

    return True

def recount_responses(response, path=""):
    false_count = 0
    added_count = 0
    different_fields = []

    if isinstance(response, dict):
        for key, value in response.items():
            current_path = f"{path}.{key}" if path else key

            false, added, fields = recount_responses(
                value,
                current_path
            )

            false_count += false
            added_count += added
            different_fields.extend(fields)

    elif isinstance(response, list):
        for i, item in enumerate(response):
            current_path = f"{path}[{i}]"

            false, added, fields = recount_responses(
                item,
                current_path
            )

            false_count += false
            added_count += added
            different_fields.extend(fields)

    elif response is False:
        false_count += 1
        different_fields.append({
            "path": path,
            "status": "removed"
        })

    elif response == "added":
        added_count += 1
        different_fields.append({
            "path": path,
            "status": "added"
        })

    return false_count, added_count, different_fields

## Sports

In [39]:
sports = explore_endpoint(
    url_v="1", 
    endpoint="sports"
)
display(sports['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'sports': [{'id': 1,
   'code': 'mlb',
   'link': '/api/v1/sports/1',
   'name': 'Major League Baseball',
   'abbreviation': 'MLB',
   'sortOrder': 11,
   'activeStatus': True},
  {'id': 11,
   'code': 'aaa',
   'link': '/api/v1/sports/11',
   'name': 'Triple-A',
   'abbreviation': 'AAA',
   'sortOrder': 101,
   'activeStatus': True},
  {'id': 12,
   'code': 'aax',
   'link': '/api/v1/sports/12',
   'name': 'Double-A',
   'abbreviation': 'AA',
   'sortOrder': 201,
   'activeStatus': True},
  {'id': 13,
   'code': 'afa',
   'link': '/api/v1/sports/13',
   'name': 'High-A',
   'abbreviation': 'A+',
   'sortOrder': 301,
   'activeStatus': True},
  {'id': 14,
   'code': 'afx',
   'link': '/api/v1/sports/14',
   'name': 'Single-A',
   'abbreviation': 'A',
   'sortOrder': 401,
   'activeStatus': True},
  {'id': 

In [40]:
# MLB: id = 1 
MLB_SPORT_ID = 1

mlb = explore_endpoint(
    url_v="1", 
    endpoint=f"sports/{MLB_SPORT_ID}"
)
display(mlb['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'sports': [{'id': 1,
   'code': 'mlb',
   'link': '/api/v1/sports/1',
   'name': 'Major League Baseball',
   'abbreviation': 'MLB',
   'sortOrder': 11,
   'activeStatus': True}]}

## Leagues

In [41]:
league = explore_endpoint(
    url_v="1",
    endpoint="league"
)
display(league['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'leagues': [{'id': 103,
   'name': 'American League',
   'link': '/api/v1/league/103',
   'abbreviation': 'AL',
   'nameShort': 'American',
   'seasonState': 'inseason',
   'hasWildCard': True,
   'hasSplitSeason': False,
   'numGames': 162,
   'hasPlayoffPoints': False,
   'numTeams': 15,
   'numWildcardTeams': 3,
   'seasonDateInfo': {'seasonId': '2026',
    'preSeasonStartDate': '2026-01-01',
    'preSeasonEndDate': '2026-02-19',
    'seasonStartDate': '2026-02-20',
    'springStartDate': '2026-02-20',
    'springEndDate': '2026-03-24',
    'regularSeasonStartDate': '2026-03-25',
    'lastDate1stHalf': '2026-07-14',
    'allStarDate': '2026-07-14',
    'firstDate2ndHalf': '2026-07-19',
    'regularSeasonEndDate': '2026-09-27',
    'postSeasonStartDate': '2026-09-28',
    'postSeasonEndDate': '2026-10-31

In [42]:
# AL: id = 103
AL_LEAGUE_ID = 103

al = explore_endpoint(
    url_v="1",
    endpoint=f"league/{AL_LEAGUE_ID}"
)
display(al['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'leagues': [{'id': 103,
   'name': 'American League',
   'link': '/api/v1/league/103',
   'abbreviation': 'AL',
   'nameShort': 'American',
   'seasonState': 'inseason',
   'hasWildCard': True,
   'hasSplitSeason': False,
   'numGames': 162,
   'hasPlayoffPoints': False,
   'numTeams': 15,
   'numWildcardTeams': 3,
   'seasonDateInfo': {'seasonId': '2026',
    'preSeasonStartDate': '2026-01-01',
    'preSeasonEndDate': '2026-02-19',
    'seasonStartDate': '2026-02-20',
    'springStartDate': '2026-02-20',
    'springEndDate': '2026-03-24',
    'regularSeasonStartDate': '2026-03-25',
    'lastDate1stHalf': '2026-07-14',
    'allStarDate': '2026-07-14',
    'firstDate2ndHalf': '2026-07-19',
    'regularSeasonEndDate': '2026-09-27',
    'postSeasonStartDate': '2026-09-28',
    'postSeasonEndDate': '2026-10-31

In [43]:
# NL: id = 104
NL_LEAGUE_ID = 104

nl = explore_endpoint(
    url_v="1",
    endpoint=f"league/{NL_LEAGUE_ID}"
)
display(nl['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'leagues': [{'id': 104,
   'name': 'National League',
   'link': '/api/v1/league/104',
   'abbreviation': 'NL',
   'nameShort': 'National',
   'seasonState': 'inseason',
   'hasWildCard': True,
   'hasSplitSeason': False,
   'numGames': 162,
   'hasPlayoffPoints': False,
   'numTeams': 15,
   'numWildcardTeams': 3,
   'seasonDateInfo': {'seasonId': '2026',
    'preSeasonStartDate': '2026-01-01',
    'preSeasonEndDate': '2026-02-19',
    'seasonStartDate': '2026-02-20',
    'springStartDate': '2026-02-20',
    'springEndDate': '2026-03-24',
    'regularSeasonStartDate': '2026-03-25',
    'lastDate1stHalf': '2026-07-14',
    'allStarDate': '2026-07-14',
    'firstDate2ndHalf': '2026-07-19',
    'regularSeasonEndDate': '2026-09-27',
    'postSeasonStartDate': '2026-09-28',
    'postSeasonEndDate': '2026-10-31

## Divisions

In [44]:
al_divisions = explore_endpoint(
    url_v="1",
    endpoint="divisions",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", AL_LEAGUE_ID)
    ]
)
display(al_divisions['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'divisions': [{'id': 200,
   'name': 'American League West',
   'season': '2026',
   'nameShort': 'AL West',
   'link': '/api/v1/divisions/200',
   'abbreviation': 'ALW',
   'league': {'id': 103, 'link': '/api/v1/league/103'},
   'sport': {'id': 1, 'link': '/api/v1/sports/1'},
   'hasWildcard': False,
   'sortOrder': 24,
   'numPlayoffTeams': 1,
   'active': True},
  {'id': 201,
   'name': 'American League East',
   'season': '2026',
   'nameShort': 'AL East',
   'link': '/api/v1/divisions/201',
   'abbreviation': 'ALE',
   'league': {'id': 103, 'link': '/api/v1/league/103'},
   'sport': {'id': 1, 'link': '/api/v1/sports/1'},
   'hasWildcard': False,
   'sortOrder': 22,
   'numPlayoffTeams': 1,
   'active': True},
  {'id': 202,
   'name': 'American League Central',
   'season': '2026',
   'nameShort': 'AL 

In [45]:
nl_divisions = explore_endpoint(
    url_v="1",
    endpoint="divisions",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", NL_LEAGUE_ID)
    ]
)
display(nl_divisions['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'divisions': [{'id': 203,
   'name': 'National League West',
   'season': '2026',
   'nameShort': 'NL West',
   'link': '/api/v1/divisions/203',
   'abbreviation': 'NLW',
   'league': {'id': 104, 'link': '/api/v1/league/104'},
   'sport': {'id': 1, 'link': '/api/v1/sports/1'},
   'hasWildcard': False,
   'sortOrder': 34,
   'numPlayoffTeams': 1,
   'active': True},
  {'id': 204,
   'name': 'National League East',
   'season': '2026',
   'nameShort': 'NL East',
   'link': '/api/v1/divisions/204',
   'abbreviation': 'NLE',
   'league': {'id': 104, 'link': '/api/v1/league/104'},
   'sport': {'id': 1, 'link': '/api/v1/sports/1'},
   'hasWildcard': False,
   'sortOrder': 32,
   'numPlayoffTeams': 1,
   'active': True},
  {'id': 205,
   'name': 'National League Central',
   'season': '2026',
   'nameShort': 'NL 

## Teams

In [46]:
al_teams = explore_endpoint(
    url_v="1",
    endpoint="teams",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", AL_LEAGUE_ID)
    ]
)
al_teams_id = {team['name']: team['id'] for team in al_teams['response']['teams']}
display(al_teams['response'])
display(al_teams_id)

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'teams': [{'springLeague': {'id': 114,
    'name': 'Cactus League',
    'link': '/api/v1/league/114',
    'abbreviation': 'CL'},
   'allStarStatus': 'N',
   'id': 133,
   'name': 'Athletics',
   'link': '/api/v1/teams/133',
   'season': 2026,
   'venue': {'id': 2529,
    'name': 'Sutter Health Park',
    'link': '/api/v1/venues/2529'},
   'springVenue': {'id': 2507, 'link': '/api/v1/venues/2507'},
   'teamCode': 'ath',
   'fileCode': 'ath',
   'abbreviation': 'ATH',
   'teamName': 'Athletics',
   'locationName': 'Sacramento',
   'firstYearOfPlay': '1901',
   'league': {'id': 103,
    'name': 'American League',
    'link': '/api/v1/league/103'},
   'division': {'id': 200,
    'name': 'American League West',
    'link': '/api/v1/divisions/200'},
   'sport': {'id': 1,
    'link': '/api/v1/sports/1',
    'name

{'Athletics': 133,
 'Seattle Mariners': 136,
 'Tampa Bay Rays': 139,
 'Los Angeles Angels': 108,
 'Texas Rangers': 140,
 'Toronto Blue Jays': 141,
 'Baltimore Orioles': 110,
 'Minnesota Twins': 142,
 'Boston Red Sox': 111,
 'Chicago White Sox': 145,
 'Cleveland Guardians': 114,
 'New York Yankees': 147,
 'Detroit Tigers': 116,
 'Houston Astros': 117,
 'Kansas City Royals': 118}

In [47]:
nl_teams = explore_endpoint(
    url_v="1",
    endpoint="teams",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("leagueId", NL_LEAGUE_ID)
    ]
)
nl_teams_id = {team['name']: team['id'] for team in nl_teams['response']['teams']}
display(nl_teams['response'])
display(nl_teams_id)

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'teams': [{'springLeague': {'id': 115,
    'name': 'Grapefruit League',
    'link': '/api/v1/league/115',
    'abbreviation': 'GL'},
   'allStarStatus': 'N',
   'id': 134,
   'name': 'Pittsburgh Pirates',
   'link': '/api/v1/teams/134',
   'season': 2026,
   'venue': {'id': 31, 'name': 'PNC Park', 'link': '/api/v1/venues/31'},
   'springVenue': {'id': 2526, 'link': '/api/v1/venues/2526'},
   'teamCode': 'pit',
   'fileCode': 'pit',
   'abbreviation': 'PIT',
   'teamName': 'Pirates',
   'locationName': 'Pittsburgh',
   'firstYearOfPlay': '1882',
   'league': {'id': 104,
    'name': 'National League',
    'link': '/api/v1/league/104'},
   'division': {'id': 205,
    'name': 'National League Central',
    'link': '/api/v1/divisions/205'},
   'sport': {'id': 1,
    'link': '/api/v1/sports/1',
    'name': 'Majo

{'Pittsburgh Pirates': 134,
 'San Diego Padres': 135,
 'San Francisco Giants': 137,
 'St. Louis Cardinals': 138,
 'Arizona Diamondbacks': 109,
 'Philadelphia Phillies': 143,
 'Chicago Cubs': 112,
 'Atlanta Braves': 144,
 'Cincinnati Reds': 113,
 'Miami Marlins': 146,
 'Colorado Rockies': 115,
 'Los Angeles Dodgers': 119,
 'Washington Nationals': 120,
 'New York Mets': 121,
 'Milwaukee Brewers': 158}

## Venue

In [48]:
pirates_venue_id = 31
pirates_venue = explore_endpoint(
    url_v="1",
    endpoint=f"venues/{pirates_venue_id}",
    params=[
        ("hydrate", "fieldInfo")
    ]
)
display(pirates_venue['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'venues': [{'id': 31,
   'name': 'PNC Park',
   'link': '/api/v1/venues/31',
   'fieldInfo': {'capacity': 38753,
    'turfType': 'Grass',
    'roofType': 'Open',
    'leftLine': 325,
    'left': 389,
    'leftCenter': 410,
    'center': 399,
    'rightCenter': 375,
    'rightLine': 320},
   'active': True,
   'season': '2026'}]}

## Roster

In [49]:
pirates_team_id = 134
pirates_team = explore_endpoint(
    url_v="1",
    endpoint=f"teams/{pirates_team_id}/roster/fullRoster"
)
display(pirates_team['response'])

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'roster': [{'person': {'id': 663336,
    'fullName': 'Aaron Shortridge',
    'link': '/api/v1/people/663336'},
   'jerseyNumber': '34',
   'position': {'code': '1',
    'name': 'Pitcher',
    'type': 'Pitcher',
    'abbreviation': 'P'},
   'status': {'code': 'A', 'description': 'Active'},
   'parentTeamId': 134},
  {'person': {'id': 815925,
    'fullName': 'Adbiel Feliz',
    'link': '/api/v1/people/815925'},
   'jerseyNumber': '34',
   'position': {'code': '6',
    'name': 'Shortstop',
    'type': 'Infielder',
    'abbreviation': 'SS'},
   'status': {'code': 'A', 'description': 'Active'},
   'parentTeamId': 134},
  {'person': {'id': 801743,
    'fullName': 'Adolfo Oviedo',
    'link': '/api/v1/people/801743'},
   'jerseyNumber': '23',
   'position': {'code': '1',
    'name': 'Pitcher',
    'type': 'Pitche

## Schedule

In [50]:
schedule = explore_endpoint(
    url_v="1",
    endpoint="schedule",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("date", "2026-08-11")
    ]
)
display(schedule['response'])
with open(f"{folder}/schedule_response.json", "w") as f:
    json.dump(schedule['response'], f, indent=4)

{'copyright': 'Copyright 2026 MLB Advanced Media, L.P.  Use of any content on this page acknowledges agreement to the terms posted here http://gdx.mlb.com/components/copyright.txt',
 'totalItems': 15,
 'totalEvents': 0,
 'totalGames': 15,
 'totalGamesInProgress': 0,
 'dates': [{'date': '2026-08-11',
   'totalItems': 15,
   'totalEvents': 0,
   'totalGames': 15,
   'totalGamesInProgress': 0,
   'games': [{'gamePk': 824240,
     'gameGuid': 'c2207627-9752-4739-98a0-68e7891d90e9',
     'link': '/api/v1.1/game/824240/feed/live',
     'gameType': 'R',
     'season': '2026',
     'gameDate': '2026-08-11T22:40:00Z',
     'officialDate': '2026-08-11',
     'status': {'abstractGameState': 'Final',
      'codedGameState': 'F',
      'detailedState': 'Final',
      'statusCode': 'F',
      'startTimeTBD': False,
      'abstractGameCode': 'F'},
     'teams': {'away': {'team': {'id': 114,
        'name': 'Cleveland Guardians',
        'link': '/api/v1/teams/114'},
       'leagueRecord': {'wins': 58

In [51]:
schedule_hidration = explore_endpoint(
    url_v="1",
    endpoint="schedule",
    params=[
        ("sportId", MLB_SPORT_ID),
        ("date", "2026-08-11"),
        ("hydrate", "team(standings)")
    ]
)
with open(f"{folder}/schedule_hydration_response.json", "w") as f:
    json.dump(schedule_hidration['response'], f, indent=4)

In [52]:
comparacion = compare_responses(
    schedule['response'],
    schedule_hidration['response']
)
recount_responses(comparacion)

(0,
 600,
 [{'path': 'dates[0].games[0].teams.away.springLeague', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.franchiseName',
   'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.clubName', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.league', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.locationName',
   'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.sport', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.springLeague',
   'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.teamName', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.springVenue', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.allStarStatus',
   'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.shortName', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.away.team.teamCode', 'status': 'added'},
  {'path': 'dates[0].games[0].teams.aw

In [53]:
GAME_PK = schedule['response']['dates'][0]['games'][0]['gamePk']
print(f"Game PK: {GAME_PK}")

Game PK: 824240


## Game

### Gumbo

In [55]:
gumbo = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live"
)
with open(f"{folder}/gumbo_schema.json", "w") as f:
    json.dump(gumbo['schema'], f, indent=4)
with open(f"{folder}/gumbo_response.json", "w") as f:
    json.dump(gumbo['response'], f, indent=4)

## Gumbo hidrations

### credits

In [56]:
gumbo_hydration_credits = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live",
    params=[
        ("hydrate", "credits")
    ]
)
with open(f"{folder}/gumbo_hydration_credits_response.json", "w") as f:
    json.dump(gumbo_hydration_credits['response'], f, indent=4)

In [57]:
comparacion = compare_responses(
    gumbo['response'],
    gumbo_hydration_credits['response']
)
recount_responses(comparacion)

(0,
 77,
 [{'path': 'liveData.plays.currentPlay.credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[0].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[1].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[2].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[3].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[4].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[5].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[6].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[7].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[8].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[9].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[10].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[11].credits', 'status': 'added'},
  {'path': 'liveData.plays.allPlays[12].credits', 'status': 'added'},
  {'path': 'liveData.p

### alignment

In [58]:
gumbo_hydration_alignment = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live",
    params=[
        ("hydrate", "alignment")
    ]
)
with open(f"{folder}/gumbo_hydration_alignment_response.json", "w") as f:
    json.dump(gumbo_hydration_alignment['response'], f, indent=4)

In [59]:
comparacion = compare_responses(
    gumbo['response'],
    gumbo_hydration_alignment['response']
)
recount_responses(comparacion)

(0,
 702,
 [{'path': 'liveData.plays.currentPlay.playEvents[0].offense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[0].defense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[1].offense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[1].defense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[2].offense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[2].defense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[3].offense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[3].defense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[4].offense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[4].defense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[5].offense',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playE

### preState

In [60]:
gumbo_hydration_preState = explore_endpoint(
    url_v="1.1",
    endpoint=f"game/{GAME_PK}/feed/live",
    params=[
        ("hydrate", "preState")
    ]
)
with open(f"{folder}/gumbo_hydration_preState_response.json", "w") as f:
    json.dump(gumbo_hydration_preState['response'], f, indent=4)

In [61]:
comparacion = compare_responses(
    gumbo['response'],
    gumbo_hydration_preState['response']
)
recount_responses(comparacion)

(0,
 354,
 [{'path': 'liveData.plays.currentPlay.playEvents[0].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[1].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[2].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[3].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[4].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[5].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.currentPlay.playEvents[6].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.allPlays[0].playEvents[0].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.allPlays[0].playEvents[1].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.allPlays[0].playEvents[2].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.allPlays[0].playEvents[3].preCount',
   'status': 'added'},
  {'path': 'liveData.plays.allPla